In [1]:
import tensorflow as tf
import tensorflow_data_validation as tfdv
import pandas as pd
import numpy as np
from tensorflow_metadata.proto.v0 import schema_pb2

print('Tf version: ', tf.__version__)
print('TFDV version: ', tfdv.__version__)



Tf version:  2.17.0
TFDV version:  1.14.0


In [106]:
# -------------------------
# 📂 Load raw data
# -------------------------
raw_df = pd.read_csv('dataset/laptop_data_1M_v2_CLEANED.csv')
print(raw_df.columns)
print(f"Loaded {len(raw_df):,} rows.")

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
       'Memory', 'Gpu', 'OpSys', 'Weight', 'Price'],
      dtype='object')
Loaded 1,000,000 rows.


In [107]:
# Select numeric columns
numeric_df = raw_df.select_dtypes(include=[np.number])

summary = pd.DataFrame({
    'min': numeric_df.min(),
    'max': numeric_df.max(),
    'mean': numeric_df.mean(),
    'median': numeric_df.median(),
    #'mode': numeric_df.mode()
})
print(summary)

             min       max          mean    median
Inches   10.1000      35.6     15.143428     15.60
Ram       1.0000      64.0      8.467997      8.00
Weight    0.0002      11.1      2.081096      2.06
Price  -500.0000  999999.0  62455.164907  52054.56


In [108]:
# -------------------------
# 🔀 Split into train, eval, test
# -------------------------
train = raw_df.sample(frac=0.8, random_state=42)
remaining_df = raw_df.drop(train_df.index)
eval = remaining_df.sample(frac=0.5, random_state=42)  # 10% of total
test = remaining_df.drop(eval_df.index)                 # 10% of total

print(
    f"Train: {len(train)}, Eval: {len(eval)}, Test: {len(test)}"
)


Train: 800000, Eval: 100000, Test: 100000


In [109]:
# Start with an empty schema
schema = schema_pb2.Schema()

# Define Price as float with value range
price_feature = schema.feature.add()
price_feature.name = 'Price'
price_feature.type = schema_pb2.FeatureType.FLOAT
price_domain = price_feature.float_domain
price_domain.min = 10000.0
price_domain.max = 300000.0

In [110]:
# Define Company as string
company_feature = schema.feature.add()
company_feature.name = 'Company'
company_feature.type = schema_pb2.FeatureType.BYTES

In [111]:
# Define TypeName as string
typename_feature = schema.feature.add()
typename_feature.name = 'TypeName'
typename_feature.type = schema_pb2.FeatureType.BYTES

In [112]:
# Define Inches as float
inches_feature = schema.feature.add()
inches_feature.name = 'Inches'
inches_feature.type = schema_pb2.FeatureType.FLOAT
inches_domain = inches_feature.float_domain
inches_domain.min = 10.0
inches_domain.max = 32.0

In [113]:
# Define ScreenResolution as string
screen_resolution_feature = schema.feature.add()
screen_resolution_feature.name = 'ScreenResolution'
screen_resolution_feature.type = schema_pb2.FeatureType.BYTES

In [114]:
# Define Ram as float
ram_feature = schema.feature.add()
ram_feature.name = 'Ram'
ram_feature.type = schema_pb2.FeatureType.FLOAT
ram_domain = ram_feature.float_domain
ram_domain.min = 1.0
ram_domain.max = 64.0

In [115]:
# Define Memory as string
memory_feature = schema.feature.add()
memory_feature.name = 'Memory'
memory_feature.type = schema_pb2.FeatureType.BYTES

In [116]:
# Define OpSys as string
opsys_feature = schema.feature.add()
opsys_feature.name = 'OpSys'
opsys_feature.type = schema_pb2.FeatureType.BYTES

In [117]:
# Define Weight as string
weight_feature = schema.feature.add()
weight_feature.name = 'Weight'
weight_feature.type = schema_pb2.FeatureType.FLOAT
weight_domain = weight_feature.float_domain
weight_domain.min = 0.5
weight_domain.max = 5.0

In [118]:
# Define Cpu as string
cpu_feature = schema.feature.add()
cpu_feature.name = 'Cpu'
cpu_feature.type = schema_pb2.FeatureType.BYTES

In [119]:
# Define Gpu as string
gpu_feature = schema.feature.add()
gpu_feature.name = 'Gpu'
gpu_feature.type = schema_pb2.FeatureType.BYTES

In [120]:
tfdv.write_schema_text(schema, 'clean_manual_schema.pbtxt')

In [121]:
# Read manual schema
schema = tfdv.load_schema_text('clean_manual_schema.pbtxt')
schema

feature {
  name: "Price"
  type: FLOAT
  float_domain {
    min: 10000.0
    max: 300000.0
  }
}
feature {
  name: "Company"
  type: BYTES
}
feature {
  name: "TypeName"
  type: BYTES
}
feature {
  name: "Inches"
  type: FLOAT
  float_domain {
    min: 10.0
    max: 32.0
  }
}
feature {
  name: "ScreenResolution"
  type: BYTES
}
feature {
  name: "Ram"
  type: FLOAT
  float_domain {
    min: 1.0
    max: 64.0
  }
}
feature {
  name: "Memory"
  type: BYTES
}
feature {
  name: "OpSys"
  type: BYTES
}
feature {
  name: "Weight"
  type: FLOAT
  float_domain {
    min: 0.5
    max: 5.0
  }
}
feature {
  name: "Cpu"
  type: BYTES
}
feature {
  name: "Gpu"
  type: BYTES
}

In [122]:
# 2️⃣ Generate statistics for each split
train_stats = tfdv.generate_statistics_from_dataframe(train)
eval_stats = tfdv.generate_statistics_from_dataframe(eval)
test_stats = tfdv.generate_statistics_from_dataframe(test)

In [123]:
# 3️⃣ Validate each split against the manual schema
train_anomalies = tfdv.validate_statistics(train_stats, schema)
eval_anomalies = tfdv.validate_statistics(eval_stats, schema)
test_anomalies = tfdv.validate_statistics(test_stats, schema)

In [124]:
# 4️⃣ Display the anomalies
print('🚀 Train anomalies:')
tfdv.display_anomalies(train_anomalies)

🚀 Train anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'__index_level_0__',New column,New column (column in data but not in schema)
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)
'Weight',Multiple errors,Unexpectedly low values: 0.0002<0.5(upto six significant digits) Unexpectedly high value: 11.1>5(upto six significant digits)


In [125]:
print('🚀 Eval anomalies:')
tfdv.display_anomalies(eval_anomalies)

🚀 Eval anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'__index_level_0__',New column,New column (column in data but not in schema)
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)
'Weight',Multiple errors,Unexpectedly low values: 0.0002<0.5(upto six significant digits) Unexpectedly high value: 11.1>5(upto six significant digits)


In [126]:
print('🚀 Test anomalies:')
tfdv.display_anomalies(test_anomalies)

🚀 Test anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'Weight',Multiple errors,Unexpectedly low values: 0.0002<0.5(upto six significant digits) Unexpectedly high value: 11.1>5(upto six significant digits)
'__index_level_0__',New column,New column (column in data but not in schema)
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)


In [127]:
from tensorflow_metadata.proto.v0 import anomalies_pb2

def clean_df_with_schema(df, schema):
    stats = tfdv.generate_statistics_from_dataframe(df)
    anomalies = tfdv.validate_statistics(stats, schema)

    for feature_name, anomaly_info in anomalies.anomaly_info.items():
        print(f"\n=== {feature_name} ===")
        #print(anomaly_info)  # full proto contents
        feature_schema = next((f for f in schema.feature if f.name == feature_name), None)
        #print(feature_schema)

        for reason in anomaly_info.reason:
            short_desc = reason.short_description.lower()
            print(reason)

            if "new column" in short_desc:
                if feature_name in df.columns:
                    print(f"🧹 Dropping new column: {feature_name}")
                    df.drop(columns=[feature_name], inplace=True)

            elif "out-of-range" in short_desc and feature_schema and feature_schema.float_domain:
                min_expected = feature_schema.float_domain.min
                max_expected = feature_schema.float_domain.max
                print(f"🧹 Clipping {feature_name} to [{min_expected}, {max_expected}]")
                df.loc[df[feature_name] < min_expected, feature_name] = min_expected
                df.loc[df[feature_name] > max_expected, feature_name] = max_expected
            else:
                print(reason)

    return df


In [128]:
train_df.columns

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
       'Memory', 'Gpu', 'OpSys', 'Weight', 'Price'],
      dtype='object')

In [129]:
train_df = clean_df_with_schema(train, schema)
train_stats = tfdv.generate_statistics_from_dataframe(train_df)
train_anomalies = tfdv.validate_statistics(train_stats, schema)
tfdv.display_anomalies(train_anomalies)


=== Inches ===
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly high value: 35.6>32(upto six significant digits)"

🧹 Clipping Inches to [10.0, 32.0]

=== Price ===
type: FLOAT_TYPE_SMALL_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly low values: -500<10000(upto six significant digits)"

🧹 Clipping Price to [10000.0, 300000.0]
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly high value: 999999>300000(upto six significant digits)"

🧹 Clipping Price to [10000.0, 300000.0]

=== __index_level_0__ ===
type: SCHEMA_NEW_COLUMN
short_description: "New column"
description: "New column (column in data but not in schema)"


=== Weight ===
type: FLOAT_TYPE_SMALL_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly low values: 0.0002<0.5(upto six significant digits)"

🧹 Clipping Weight to [0.5, 5.0]
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range va

,Anomaly short description,Anomaly long description
Feature name,,
'__index_level_0__',New column,New column (column in data but not in schema)


In [130]:
eval_df = clean_df_with_schema(eval, schema)
eval_stats = tfdv.generate_statistics_from_dataframe(eval_df)
eval_anomalies = tfdv.validate_statistics(eval_stats, schema)
tfdv.display_anomalies(eval_anomalies)


=== Inches ===
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly high value: 35.6>32(upto six significant digits)"

🧹 Clipping Inches to [10.0, 32.0]

=== Price ===
type: FLOAT_TYPE_SMALL_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly low values: -500<10000(upto six significant digits)"

🧹 Clipping Price to [10000.0, 300000.0]
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly high value: 999999>300000(upto six significant digits)"

🧹 Clipping Price to [10000.0, 300000.0]

=== Weight ===
type: FLOAT_TYPE_SMALL_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly low values: 0.0002<0.5(upto six significant digits)"

🧹 Clipping Weight to [0.5, 5.0]
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly high value: 11.1>5(upto six significant digits)"

🧹 Clipping Weight to [0.5, 5.0]

=== __index_level_0__ ===
typ

,Anomaly short description,Anomaly long description
Feature name,,
'__index_level_0__',New column,New column (column in data but not in schema)


In [131]:
test_df = clean_df_with_schema(test, schema)
test_stats = tfdv.generate_statistics_from_dataframe(test_df)
test_anomalies = tfdv.validate_statistics(test_stats, schema)
tfdv.display_anomalies(test_anomalies)


=== Price ===
type: FLOAT_TYPE_SMALL_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly low values: -500<10000(upto six significant digits)"

🧹 Clipping Price to [10000.0, 300000.0]
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly high value: 999999>300000(upto six significant digits)"

🧹 Clipping Price to [10000.0, 300000.0]

=== Inches ===
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly high value: 35.6>32(upto six significant digits)"

🧹 Clipping Inches to [10.0, 32.0]

=== Weight ===
type: FLOAT_TYPE_SMALL_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly low values: 0.0002<0.5(upto six significant digits)"

🧹 Clipping Weight to [0.5, 5.0]
type: FLOAT_TYPE_BIG_FLOAT
short_description: "Out-of-range values"
description: "Unexpectedly high value: 11.1>5(upto six significant digits)"

🧹 Clipping Weight to [0.5, 5.0]

=== __index_level_0__ ===
typ

,Anomaly short description,Anomaly long description
Feature name,,
'__index_level_0__',New column,New column (column in data but not in schema)


In [132]:
train_df.to_csv("dataset/train_cleaned.csv", index=False)
eval_df.to_csv("dataset/eval_cleaned.csv", index=False)
test_df.to_csv("dataset/test_cleaned.csv", index=False)

In [133]:
for dataset in ["train", "eval", "test"]:
    df = pd.read_csv(f'dataset/{dataset}_cleaned.csv')
    numeric_df = df.select_dtypes(include=[np.number])

    summary = pd.DataFrame({
        'min': numeric_df.min(),
        'max': numeric_df.max(),
        'mean': numeric_df.mean(),
        'median': numeric_df.median()
    })
    print(summary)

            min       max          mean    median
Inches     10.1      32.0     15.134918     15.60
Ram         1.0      64.0      8.465588      8.00
Weight      0.5       5.0      2.066989      2.06
Price   10000.0  300000.0  60181.385534  52054.56
            min       max          mean    median
Inches     10.1      32.0     15.130932     15.60
Ram         1.0      64.0      8.505412      8.00
Weight      0.5       5.0      2.066927      2.06
Price   10000.0  300000.0  60325.038014  52054.56
            min       max          mean    median
Inches     10.1      32.0     15.128757     15.60
Ram         1.0      64.0      8.449853      8.00
Weight      0.5       5.0      2.064327      2.06
Price   10000.0  300000.0  59937.349009  52054.56
